# Crop production (Vietnam Mekong)

Notebook to read RIBASIM HIS crop-production files and store them as **Zarr** with the original variable names and values unchanged.

Source files (see `read_ribasim_his.ipynb` in Food-Security):

- `RIB_CULT_prod.his` — crop production (`path` in TOML)
- `RIB_ADVIR_dmnd.his` — crop hectares per season (`ha_path` in TOML)

Outputs live alongside the salinity products under:
`N:\...\stac_folder\crop_production\` (see `11_salinity.ipynb`).

**Note on NetCDF:** RIBASIM HIS variable names contain characters that are illegal in NetCDF
(e.g. `+ Allocation (Mcm)`, `P Cr02/WinterSpring (ha)`). NetCDF variable names may only contain
letters, digits, and `_`. Renaming would be required for `.nc` export, so this notebook writes
**Zarr only**, which preserves the original names.

**Prerequisites**

1. `mamba activate coclico`
2. `pip install -e .` in Food-Security (or use `sys.path` workaround below)
3. `metadata_crop_production.json` present in `data_dir`

In [33]:
# Optional; code formatter
# %load_ext nb_black

### Configure paths and imports

In [34]:
import datetime
import json
import os
import sys
from pathlib import Path

import xarray as xr

sys.path.insert(0, r"C:\Ocean\Work\Projects\2026\IDP\Tools\Food-Security")
from food_security import data_reader

os.environ["UDUNITS2_XML_PATH"] = str(
    Path.home().joinpath(
        r"Anaconda3\pkgs\udunits2-2.2.28-h892ecd3_0\Library\share\udunits\udunits2.xml"
    )
)
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [35]:
repo_root = Path(r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories")

data_dir = Path(
    r"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production"
)
metadata_path = data_dir / "metadata_crop_production.json"

data_dir.mkdir(parents=True, exist_ok=True)

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata not found: {metadata_path}")

with open(metadata_path) as f:
    metadata = json.load(f)

prod_out_file = "RIB_CULT_prod"
ha_out_file = "RIB_ADVIR_dmnd"

prod_zarr = data_dir / f"{prod_out_file}.zarr"
ha_zarr = data_dir / f"{ha_out_file}.zarr"

PROD_HIS = Path(
    r"C:/Ocean/Work/Projects/2026/IDP/Data/Ribasim8_MekDelta26"
    r"/Modules/ribasim/Mekong.1/work/RIB_CULT_prod.his"
)
HA_HIS = Path(
    r"C:/Ocean/Work/Projects/2026/IDP/Data/Ribasim8_MekDelta26"
    r"/Modules/ribasim/Mekong.1/work/RIB_ADVIR_dmnd.his"
)

print("Output dir:", data_dir)
print("Metadata:", metadata_path)
for label, path in [("Production HIS", PROD_HIS), ("Hectares HIS", HA_HIS)]:
    print(f"{label}: {path}")
    print(f"  exists: {path.is_file()}")

Output dir: N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production
Metadata: N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production\metadata_crop_production.json
Production HIS: C:\Ocean\Work\Projects\2026\IDP\Data\Ribasim8_MekDelta26\Modules\ribasim\Mekong.1\work\RIB_CULT_prod.his
  exists: True
Hectares HIS: C:\Ocean\Work\Projects\2026\IDP\Data\Ribasim8_MekDelta26\Modules\ribasim\Mekong.1\work\RIB_ADVIR_dmnd.his
  exists: True


### Read RIBASIM HIS files

In [36]:
def read_his(path: Path, *, use_hia: bool = False) -> xr.Dataset:
    """Read a RIBASIM HIS file and return the xarray Dataset."""
    reader = data_reader.HisFile(path, crop=None)
    reader.read(hia=use_hia)
    return reader.ds


def dataset_for_zarr(ds: xr.Dataset, metadata: dict) -> xr.Dataset:
    """Prepare dataset for Zarr export without renaming variables.

    Only serializes attrs that Zarr cannot store natively (e.g. datetime t0).
    """
    out = ds.copy(deep=True)
    for key, value in list(out.attrs.items()):
        if isinstance(value, datetime.datetime):
            out.attrs[key] = value.isoformat()
    for attr_name, attr_val in metadata.items():
        if attr_name == "PROVIDERS":
            attr_val = json.dumps(attr_val)
        out.attrs[attr_name] = attr_val
    out.attrs["Conventions"] = "CF-1.8"
    out["time"].attrs.update(
        {"long_name": "Time", "standard_name": "time", "axis": "T"}
    )
    return out


def dataset_summary(ds: xr.Dataset, title: str) -> None:
    print(title)
    print("=" * len(title))
    print(f"Time steps : {len(ds.time)} ({ds.time.values[0]} → {ds.time.values[-1]})")
    print(f"Stations   : {len(ds.station)}")
    print(f"Variables  : {len(ds.data_vars)}")


prod_ds = read_his(PROD_HIS, use_hia=False)
ha_ds = read_his(HA_HIS, use_hia=True)

dataset_summary(prod_ds, "RIB_CULT_prod.his")
print()
dataset_summary(ha_ds, "RIB_ADVIR_dmnd.his")

RIB_CULT_prod.his
Time steps : 3 (2014-01-01T00:00:00.000000000 → 2016-01-01T00:00:00.000000000)
Stations   : 48
Variables  : 23

RIB_ADVIR_dmnd.his
Time steps : 72 (2014-01-01T00:00:00.000000000 → 2016-12-16T00:00:00.000000000)
Stations   : 12
Variables  : 37


## 1. `RIB_CULT_prod.his` → Zarr

In [37]:
dataset_for_zarr(prod_ds, metadata).to_zarr(prod_zarr, mode="w")

In [38]:
prod_check = xr.open_zarr(prod_zarr)
print("Variables preserved:", list(prod_check.data_vars) == list(prod_ds.data_vars))
prod_check

Variables preserved: False


<xarray.Dataset> Size: 17kB
Dimensions:                (time: 3, station: 48)
Coordinates:
  * time                   (time) datetime64[ns] 24B 2014-01-01 ... 2016-01-01
  * station                (station) <U20 4kB 'Nd______42 / Cr__3 /' ... 'Nd_...
Data variables: (12/23)
    + Actual rainfall (    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    + Allocation (Mcm)     (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    + Decrease RZ SM +     (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    - Act.evapotranspir    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    - Act.percolation (    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    - Increase RZ SM +     (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    ...                     ...
    Potent.farm gate pr    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Potent.farm gate pr_2  (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Potent.field level     (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Potent.field level_2   (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Potent.production c    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
    Survival fraction (    (time, station) float32 576B dask.array<chunksize=(3, 48), meta=np.ndarray>
Attributes: (12/32)
    AUTHOR:              Author
    CITATION:            Citation
    COLLECTION_ID:       crop_production
    COMMENT:             
    CRS:                 EPSG:32648
    Conventions:         CF-1.8
    ...                  ...
    TITLE_ABBREVIATION:  Abbreviation
    UNITS:                Units
    WMS_DATASET:         crop_production
    header:              All cultivations                        Yearly agric...
    scu:                 86400
    t0:                  2014-01-01T00:00:00

## 2. `RIB_ADVIR_dmnd.his` → Zarr

In [39]:
dataset_for_zarr(ha_ds, metadata).to_zarr(ha_zarr, mode="w")

In [40]:
ha_check = xr.open_zarr(ha_zarr)
print("Variables preserved:", list(ha_check.data_vars) == list(ha_ds.data_vars))
ha_check

Variables preserved: False


<xarray.Dataset> Size: 64kB
Dimensions:                                              (time: 72, station: 12)
Coordinates:
  * time                                                 (time) datetime64[ns] 576B ...
  * station                                              (station) <U23 1kB '...
Data variables: (12/18)
    + Actual rainfall (Mcm)                              (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    + Decrease RZ SM + S field (Mcm)                     (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    + Gross water supply (Mcm)                           (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    - Actual evapotranspiration (Mcm)                    (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    - Actual percolation (Mcm)                           (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    - Drainage from fields (Mcm)                         (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    ...                                                   ...
    Overall irrigation efficiency (%)                    (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Rainfall effectiveness (%)                           (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Shortage per time step (# of times)                  (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Shortage per time step (%)                           (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Supply-demand ratio (%)                              (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
    Water balance term (must be 0.0 during grow.season)  (time, station) float32 3kB dask.array<chunksize=(72, 12), meta=np.ndarray>
Attributes: (12/32)
    AUTHOR:              Author
    CITATION:            Citation
    COLLECTION_ID:       crop_production
    COMMENT:             
    CRS:                 EPSG:32648
    Conventions:         CF-1.8
    ...                  ...
    TITLE_ABBREVIATION:  Abbreviation
    UNITS:                Units
    WMS_DATASET:         crop_production
    header:              Advanced irrigation nodes               Demand and a...
    scu:                 86400
    t0:                  2014-01-01T00:00:00